In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.preprocessing import PowerTransformer,FunctionTransformer
from scipy.stats import probplot
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score,r2_score
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LinearRegression,LogisticRegression
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,LabelEncoder
from sklearn import pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
df=pd.read_csv('/content/train.csv')

In [ ]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [ ]:
df.drop(columns=['Cabin','PassengerId','Ticket','Name'],inplace=True)

In [ ]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [ ]:
df.isnull().sum()

,0
Survived,0
Pclass,0
Sex,0
Age,177
SibSp,0
Parch,0
Fare,0
Embarked,2


In [ ]:
X=df.drop(columns=['Survived'])
Y=df['Survived']

In [ ]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [ ]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [ ]:
Y_train.head()

,Survived
331,0
733,0
382,0
704,0
813,0


In [ ]:
trf1=ColumnTransformer([('impute_age',SimpleImputer(),[2]),('impute_embarked',SimpleImputer(strategy='most_frequent'),[6])],remainder='passthrough')

In [ ]:
trf1.fit_transform(X_train)[0]

array([45.5, 'S', 1, 'male', 0, 0, 28.5], dtype=object)

In [ ]:
trf2=ColumnTransformer([('ohe_sex_embarked',OneHotEncoder(sparse_output=False,handle_unknown='ignore',drop='first'),[1,3]),('log',PowerTransformer(),[2,4,5,6])],remainder='passthrough')

In [ ]:
trf3=ColumnTransformer([('scale',StandardScaler(),[1,6])],remainder='passthrough')

In [ ]:
trf4=DecisionTreeClassifier()
trf4_1=LogisticRegression()
trf4_2=LinearRegression()

In [ ]:
pipe1=Pipeline([('trf1',trf1),('trf2',trf2),('trf3',trf3),('trf4',trf4)])
pipe2=Pipeline([('trf1',trf1),('trf2',trf2),('trf3',trf3),('trf4',trf4_1)])
pipe3=Pipeline([('trf1',trf1),('trf2',trf2),('trf3',trf3),('trf4',trf4_2)])

In [ ]:
pipe1.fit(X_train,Y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 3]),
                                                 ('log', PowerTransformer(),
                                                  [2, 4, 5, 6])])),
                ('trf3',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('scale', StandardScaler(),
                                                  [1, 6])])),
                ('trf4', DecisionTreeClassifier())])

In [ ]:
Y_pred1=pipe1.predict(X_test)

In [ ]:
acc1=accuracy_score(Y_test,Y_pred1)

In [ ]:
pipe2.fit(X_train,Y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 3]),
                                                 ('log', PowerTransformer(),
                                                  [2, 4, 5, 6])])),
                ('trf3',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('scale', StandardScaler(),
                                                  [1, 6])])),
                ('trf4', LogisticRegression())])

In [ ]:
Y_pred2=pipe2.predict(X_test)

In [ ]:
acc2=accuracy_score(Y_test,Y_pred2)

In [ ]:
pipe3.fit(X_train,Y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 3]),
                                                 ('log', PowerTransformer(),
                                                  [2, 4, 5, 6])])),
                ('trf3',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('scale', StandardScaler(),
                                                  [1, 6])])),
                ('trf4', LinearRegression())])

In [ ]:
Y_pred3=pipe3.predict(X_test)

In [ ]:
acc3=r2_score(Y_test,Y_pred3)

In [ ]:
print(acc1,acc2,acc3)

0.776536312849162 0.8044692737430168 0.4327409303040758


In [ ]:
np.mean(cross_val_score(pipe1,X_train,Y_train,cv=5,scoring='accuracy'))

np.float64(0.745769723234512)

In [ ]:
np.mean(cross_val_score(pipe2,X_train,Y_train,cv=5,scoring='accuracy'))

np.float64(0.7836600019698612)

In [ ]:
np.mean(cross_val_score(pipe3,X_train,Y_train,cv=5,scoring='r2'))

np.float64(0.360349173593693)

#**PIPE 2 WINS**

In [ ]:
import pickle

In [ ]:
pickle.dump(pipe2,open('LogModel.pkl','wb'))